# TIFM QLoRA Fine-Tuning on Colab
Fine-tune Qwen 2.5 7B Instruct on TIFM telecom analytics Q&A dataset.
Uses free T4 GPU (16GB VRAM). Runs in ~20 minutes.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q torch transformers accelerate peft bitsandbytes datasets trl

In [ ]:
import json
import random
import itertools
from datetime import datetime, timedelta
from pathlib import Path

random.seed(42)

SYSTEM_PROMPT = (
    'You are a Telecom Intelligence Assistant \u2014 an expert digital forensics investigator '
    'trained to analyze Call Detail Records (CDR), IP Detail Records (IPDR), and TIFM '
    'multi-agent analytics. Your role is to interpret structured analytics data and provide '
    'clear, evidence-driven answers to investigation questions. Always cite specific metrics, '
    'highlight anomalies, and assess confidence levels. Use professional forensics language.'
)

def pick(arr):
    return arr[random.randint(0, len(arr) - 1)]

def synthetic_analytics(subject_count=4, app_count=3, meeting_count=2,
                        sim_swap_count=0, device_change_count=0,
                        burner_scores=None, roles=None, communities=None):
    burner_scores = burner_scores or [5, 10, 15, 8]
    roles = roles or ['Kingpin / Coordinator', 'Hub Node / Lieutenant', 'Member', 'Bridge Node / Broker']
    subjects = [f'+91999999990{i}' for i in range(1, subject_count + 1)]
    network_nodes = {}
    for i, sub in enumerate(subjects):
        deg = round(random.uniform(0.1, 0.9), 3)
        bet = round(random.uniform(0.05, 0.7), 3)
        network_nodes[sub] = {'degree_centrality': deg, 'betweenness_centrality': bet,
                              'inferred_role': roles[i] if i < len(roles) else 'Member'}
    identity = {}
    for i, sub in enumerate(subjects):
        sim_swaps, device_changes = [], []
        for s in range(sim_swap_count if i == 0 else 0):
            sim_swaps.append({'timestamp': (datetime.now() - timedelta(days=s*2)).isoformat(),
                'imei': f'860000000{i+1:04d}', 'old_imsi': f'4044500000{(i+1)*10 + s*2}',
                'new_imsi': f'4044500000{(i+1)*10 + s*2 + 1}', 'type': 'SIM Swap', 'confidence': 'High'})
        for d in range(device_change_count if i == 1 else 0):
            device_changes.append({'timestamp': (datetime.now() - timedelta(days=d*3)).isoformat(),
                'imsi': f'4044500000{(i+1)*10}', 'old_imei': f'860000000{i+1:04d}',
                'new_imei': f'860000000{i+1+5:04d}', 'type': 'Device Change', 'confidence': 'High'})
        identity[sub] = {
            'unique_identities_count': 1 + len(sim_swaps) + len(device_changes),
            'unique_pairs': [f'IMSI: 4044500000{(i+1)*10} / IMEI: 860000000{i+1:04d}'],
            'sim_swaps': sim_swaps, 'device_changes': device_changes,
            'burner_score': burner_scores[i] if i < len(burner_scores) else 5,
            'is_suspected_burner': (burner_scores[i] if i < len(burner_scores) else 5) > 30}
    towers = ['TWR_DEL_01', 'TWR_DEL_02', 'TWR_DEL_03', 'TWR_MUM_01', 'TWR_MUM_02']
    movement = {}
    for i, sub in enumerate(subjects):
        movement[sub] = {'home_tower': towers[0], 'work_tower': towers[1],
            'total_towers_visited': random.randint(1, len(towers)),
            'towers': towers[:random.randint(1, len(towers))],
            'mobility_index': pick(['High', 'Medium', 'Low'])}
    meetings = []
    meeting_pairs = list(itertools.combinations(subjects[:min(4, len(subjects))], 2))
    for m in range(min(meeting_count, len(meeting_pairs))):
        a, b = meeting_pairs[m]; base = datetime.now() - timedelta(days=m)
        meetings.append({'tower_id': towers[m % len(towers)], 'subject_a': a, 'subject_b': b,
            'time_a': base.isoformat(), 'time_b': (base + timedelta(minutes=random.randint(5,55))).isoformat(),
            'gap_minutes': round(random.uniform(3, 55), 2)})
    apps_map = {'WhatsApp': 443, 'Telegram': 5222, 'Signal': 3478, 'VPN': 1194, 'Web': 80}
    attribution = {}
    for app in list(apps_map.keys())[:app_count]:
        attribution[app] = {'count': random.randint(5, 80), 'confidence': random.randint(70, 98),
            'evidence': [f'Port match: {apps_map[app]}', f'Protocol: {pick(["TCP", "UDP"])}']}
    mid = len(subjects) // 2
    comm_list = []
    if communities is None:
        if mid > 0: comm_list.append({'id': 1, 'members': subjects[:mid]})
        if len(subjects) > mid: comm_list.append({'id': 2, 'members': subjects[mid:]})
    else:
        comm_list = [{'id': i+1, 'members': c} for i, c in enumerate(communities)]
    return {'attribution': attribution, 'identity': identity,
        'movement': {'subjects_movement': movement, 'detected_meetings': meetings},
        'network': {'nodes_count': len(subjects), 'edges_count': random.randint(10, 40),
            'network_density': round(random.uniform(0.05, 0.6), 3),
            'centrality_metrics': network_nodes, 'communities': comm_list}}

def fmt_analytics(a):
    return json.dumps(a, indent=2)

def make_example(analytics, question, answer):
    return {'system': SYSTEM_PROMPT,
        'user': f'TIFM Analytics:\n{fmt_analytics(analytics)}\n\nInvestigator Question: {question}',
        'assistant': answer}

examples = []

for i in range(8):  # Type 1 - Subject role
    sub = f'+91999999990{i%4+1}'; role = pick(['Kingpin / Coordinator', 'Hub Node / Lieutenant', 'Bridge Node / Broker', 'Member'])
    a = synthetic_analytics(subject_count=4, roles=[role, 'Member', 'Member', 'Bridge Node / Broker'])
    q = f'What is the role of subject {sub} in this communication network?'
    ans = (f'Based on the TIFM network centrality analysis, subject **{sub}** has a degree centrality '
        f'of **{a["network"]["centrality_metrics"][sub]["degree_centrality"]}** and a betweenness '
        f'centrality of **{a["network"]["centrality_metrics"][sub]["betweenness_centrality"]}**. '
        f'Their inferred role is **{role}**. ')
    if 'Kingpin' in role:
        ans += 'This subject controls a significant portion of the communication flow. Interdiction of this node would maximally disrupt the network.'
    elif 'Hub' in role:
        ans += 'This subject serves as a lieutenant or middle-manager. Monitoring their communications may reveal leadership instructions.'
    elif 'Bridge' in role:
        ans += 'This subject is a broker connecting different sub-networks. They may be the only link between two otherwise separate groups.'
    else:
        ans += 'This subject is a regular network member. Their removal would have limited structural impact.'
    examples.append(make_example(a, q, ans))

for i in range(8):  # Type 2 - Identity/SIM swap
    has_swaps = i % 2 == 0; has_dev = i % 3 == 0
    a = synthetic_analytics(subject_count=4, sim_swap_count=3 if has_swaps else 0,
        device_change_count=2 if has_dev else 0, burner_scores=[45 if has_swaps else 10, 5, 5, 12])
    q = 'Analyze the identity profile of subject +919999999901. Are there SIM swaps, device changes, or burner indicators?'
    id_d = a['identity']['+919999999901']
    if has_swaps:
        swaps_d = '; '.join(f'on {s["timestamp"]}: IMSI changed from {s["old_imsi"]} to {s["new_imsi"]}' for s in id_d['sim_swaps'])
        ans = (f'Subject +919999999901 shows clear identity evasion behavior:\n- SIM Swaps: {len(id_d["sim_swaps"])} instances - {swaps_d}\n'
            f'- Burner Score: {id_d["burner_score"]}/100\n- Unique identity pairs: {id_d["unique_identities_count"]}\n\n'
            'Multiple SIM swaps on the same device indicate operational security awareness. Recommended: correlate swap events with changes in communication patterns.')
        if has_dev:
            ans += f'\nAdditionally, {len(id_d["device_changes"])} device changes detected - the subject is also swapping handsets.'
    else:
        ans = f'Subject +919999999901 has a clean identity profile: no SIM swaps, no device changes, burner score {id_d["burner_score"]}/100. Single identity pair used consistently.'
    examples.append(make_example(a, q, ans))

for i in range(8):  # Type 3 - Meeting
    a = synthetic_analytics(subject_count=4, meeting_count=2+i%3)
    meetings = a['movement']['detected_meetings']
    if meetings:
        m = meetings[0]
        q = f'Explain the meeting between {m["subject_a"]} and {m["subject_b"]}.'
        ans = (f'A meeting was detected between {m["subject_a"]} and {m["subject_b"]} at tower {m["tower_id"]} '
            f'with a gap of {m["gap_minutes"]} minutes. This sub-1-hour co-location is highly suggestive of an in-person meeting. '
            'Recommended: cross-reference meeting time with other intelligence, check call records around the meeting time.')
        examples.append(make_example(a, q, ans))

for i in range(8):  # Type 4 - Movement
    sub = f'+91999999990{i%4+1}'; mobility = pick(['High', 'Medium', 'Low'])
    a = synthetic_analytics(subject_count=4)
    a['movement']['subjects_movement'][sub] = {'home_tower': 'TWR_DEL_01', 'work_tower': 'TWR_DEL_02',
        'total_towers_visited': 5 if mobility == 'High' else 2, 'mobility_index': mobility,
        'towers': ['TWR_DEL_01', 'TWR_DEL_02', 'TWR_DEL_03', 'TWR_MUM_01'][:5 if mobility == 'High' else 2]}
    q = f'Describe the movement and activity patterns of {sub}.'
    ans = f'Subject {sub} demonstrates a {mobility.lower()} mobility profile with {a["movement"]["subjects_movement"][sub]["total_towers_visited"]} distinct towers.'
    examples.append(make_example(a, q, ans))

random.shuffle(examples)
print(f'Generated {len(examples)} examples')

with open('tifm_train.jsonl', 'w') as f:
    for ex in examples:
        f.write(json.dumps(ex, ensure_ascii=False) + '\n')

In [ ]:
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer

# Config
MODEL_ID = 'Qwen/Qwen2.5-7B-Instruct'
OUTPUT_DIR = 'tifm_lora_output'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True
)
lora_config = LoraConfig(
    r=16, lora_alpha=32,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    lora_dropout=0.05, bias='none', task_type='CAUSAL_LM'
)
training_args = TrainingArguments(
    per_device_train_batch_size=2, gradient_accumulation_steps=4,
    warmup_steps=50, max_steps=500, learning_rate=2e-4, fp16=True,
    logging_steps=10, save_steps=100, eval_strategy='no',
    output_dir=OUTPUT_DIR, optim='paged_adamw_8bit', save_total_limit=2,
    report_to='none'
)

# Load
print('Loading tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

print('Loading model (4-bit)...')
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=bnb_config, device_map='auto', trust_remote_code=True
)
model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, lora_config)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f'Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)')

# Dataset
dataset = load_dataset('json', data_files='tifm_train.jsonl', split='train')
def fmt(batch):
    texts = []
    for s, u, a in zip(batch['system'], batch['user'], batch['assistant']):
        msgs = [{'role': 'system', 'content': s}, {'role': 'user', 'content': u}, {'role': 'assistant', 'content': a}]
        texts.append(tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False))
    return {'text': texts}
dataset = dataset.map(fmt, batched=True)

# Train
trainer = SFTTrainer(
    model=model, train_dataset=dataset, dataset_text_field='text',
    max_seq_length=4096, tokenizer=tokenizer, args=training_args
)
print('Starting training...')
trainer.train()
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f'Model saved to {OUTPUT_DIR}')

In [ ]:
# Zip the LoRA adapter for download
import shutil
shutil.make_archive('tifm_lora_output', 'zip', 'tifm_lora_output')
print('Created tifm_lora_output.zip')

from google.colab import files
files.download('tifm_lora_output.zip')